# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [2]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [3]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [4]:
system_message = "You are a helpful assistant for an Online udemy course 'Become an LLM Engineer in 8 weeks: Build and deploy 8 LLM apps, mastering Generative AI, RAG, LoRA and AI Agents.'"
system_message += "Help student with any doubt regarding course content and assignment. Also try to help student if there is coding or environment setup issue."
system_message += "If you're unsure about something, be clear about it and recommend that the student seek help from the course instructors or support staff."

In [5]:
course_data = {
    "week 1": "Dive into the fundamentals of Transformers.Experiment with six leading Frontier Models.Build your first business Gen AI product that scrapes the web, makes decisions, and creates formatted sales brochures.",
    "week 2": "Explore Frontier APIs and interact with three leading models. Develop a customer service chatbot with a sharp UI that can interact with text, images, audio, and utilize tools or agents.",
    "week 3": "• Discover the world of Open-Source models using HuggingFace.Tackle 10 common Gen AI use cases, from translation to image generation.Build a product to generate meeting minutes and action items from recordings."
}

course_details = {
    "Creator": "Ed Donner, https://www.linkedin.com/in/eddonner/",
    "Duration": "8 Weeks",
    "Requirements":"Familiarity with Python. We recommend that you allocate around $5 for API costs to work with frontier models.(No Nessasary)",
    "Overview":"Accelerate your career in AI with practical, real-world projects led by industry veteran Ed Donner. Build advanced Generative AI products, experiment with over 20 groundbreaking models, and master state-of-the-art techniques like RAG, QLoRA, and Agents."
}

def get_course_section_details(course_section):
    section = course_section.lower()
    return course_data.get(section, "Refer Course Details Page 'www.course_details_page.com'")

def get_course_details(query):
    return course_details.get(query, "Refer Course Details Page 'www.course_details_page.com'")


course_section_details_function = {
    "name": "get_course_section_details",
    "description": "Get the details for that specific week of the course. Call this whenever you need details about specific week of the course, for example when a customer asks 'What are we doing in Week 1 of the course? '",
    "parameters": {
        "type": "object",
        "properties": {
            "course_section": {
                "type": "string",
                "description": "The is what the user what to know about the specific week of the course",
            },
        },
        "required": ["course_section"],
        "additionalProperties": False
    }
}

course_details_function = {
    "name": "get_course_details",
    "description": "Get the details regarding the course. Call this whenever user request details regarding the course, for example: 'Who made this course ?'",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "This handle details regaring Creator, Duration, Requirements, Overview",
            },
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [6]:
tools = [{"type": "function", "function": course_section_details_function}, {"type": "function", "function": course_details_function}]

In [7]:
def chat(message, history):
    messages = [{"role":"system", "content":system_message}] + history + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        handler_response = handle_tool_call(message)
        if handler_response:
            messages.append(message)
            messages.append(handler_response)
            response = openai.chat.completions.create(model=MODEL, messages=messages)
            
    return response.choices[0].message.content
            

In [8]:
def handle_tool_call(method_called):
    tool_call = method_called.tool_calls[0]
    fuction_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    response = None
    if fuction_name == "get_course_section_details":
        course_section = arguments.get('course_section')
        print("Method Called get_course_section_details ::", course_section)
        course_section_details = get_course_section_details(course_section)
        response = {
            "role": "tool",
            "content": json.dumps({"course_section": course_section,"course_section_details": course_section_details}),
            "tool_call_id": tool_call.id
        }
    if fuction_name == "get_course_details":
        query = arguments.get('query')
        print("Method Called get_course_details ::", query)
        course_details = get_course_details(query)
        response = {
            "role": "tool",
            "content": json.dumps({"query": query,"course_details": course_details}),
            "tool_call_id": tool_call.id
        }
    return response

In [9]:
# gr.ChatInterface(fn=chat, type="messages").launch()

In [10]:
# !pip install pyaudio

In [11]:
# import pyaudio
# import wave
# def record_audio():
#     # Settings
#     try:
#         FORMAT = pyaudio.paInt16
#         CHANNELS = 1
#         RATE = 44100
#         CHUNK = 1024
#         RECORD_SECONDS = 5
#         OUTPUT_FILENAME = "output.wav"
        
#         # Initialize PyAudio
#         audio = pyaudio.PyAudio()
        
#         # Start Recording
#         stream = audio.open(format=FORMAT, channels=CHANNELS,
#                             rate=RATE, input=True,
#                             frames_per_buffer=CHUNK)
        
#         print("Recording...")
#         frames = []
        
#         for _ in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
#             data = stream.read(CHUNK)
#             frames.append(data)
        
#         print("Finished recording.")
        
#         # Stop and close the stream
#         stream.stop_stream()
#         stream.close()
#         audio.terminate()
        
#         # Save to a WAV file
#         with wave.open(OUTPUT_FILENAME, 'wb') as wf:
#             wf.setnchannels(CHANNELS)
#             wf.setsampwidth(audio.get_sample_size(FORMAT))
#             wf.setframerate(RATE)
#             wf.writeframes(b''.join(frames))
#         return True
        
#     except Exception as err:
#         print("Error in audio recording module ::", err)
#         return False

In [12]:
from faster_whisper import WhisperModel

whisper_model = WhisperModel("base")

def transcribe_and_ask_gpt(audio_filepath):
    if audio_filepath is None:
        # history += [{"role":"user", "content":"No audio input received."}]
        # return history
        return "No audio input received."

    # Transcribe speech to text
    
    segments, _ = whisper_model.transcribe(audio_filepath)
    user_input = " ".join([segment.text for segment in segments]).strip()
    if not user_input:
        # history += [{"role":"user", "content":"Sorry, I couldn't understand that."}]
        msg = "Sorry, I couldn't understand that."
    else:
        # history += [{"role":"user", "content":user_input}]
        msg = user_input
    
    return msg

In [13]:
# !pip uninstall whisper
# !pip install faster-whisper

In [14]:


with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        transcript_box = gr.Textbox(label="Transcript", lines=5)
    with gr.Row():
        mic_input = gr.Audio(sources=["microphone"], type="filepath", recording=True, label="Click to Record")
        # audio_input = gr.Audio(source="microphone", type="filepath", label="Speak Here")
        # mic = gr.Microphone(label="Click to Record", type="filepath")
    with gr.Row():
        clear = gr.Button("Clear")

    def do_entry(message, history):
        history += [{"role":"user", "content":message}]
        return "", history

    # mic.change(
    #     fn=transcribe_and_ask_gpt,
    #     inputs=[mic, chatbot],
    #     outputs=[chatbot]
    # )
    mic_input.change(
        fn = transcribe_and_ask_gpt,
        inputs = mic_input,
        outputs = transcript_box
    )
    # )
    # # audio_input.change(fn=transcribe_and_ask_gpt, inputs=[audio_input, chatbot], outputs=[entry, chatbot]).then(
    # #     chat, inputs=chatbot, outputs=[chatbot, image_output]
    # # )
    
    # entry.submit(do_entry, inputs=[entry, chatbot], outputs=[entry, chatbot]).then(
    #     chat, inputs=chatbot, outputs=[chatbot, image_output]
    # )
    clear.click(lambda: None, inputs=None, outputs=chatbot, queue=False)

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [15]:
# !pip install --upgrade gradio